In [10]:
import gzip
import json
import random

# Configuration
input_file = ("Movies_and_TV.csv.gz")
output_file = "Movies_and_TV_1_percent.csv"
sample_rate = 0.01  # 1%

def sample_dataset(input_path, output_path, rate):
    count = 0
    sampled_count = 0
    
    # Opening as 'rt' (read text) handles the decompression on-the-fly
    with gzip.open(input_path, 'rt', encoding='utf-8') as f_in:
        with open(output_path, 'w', encoding='utf-8') as f_out:
            for line in f_in:
                count += 1
                # Randomly decide to keep this line
                if random.random() < rate:
                    f_out.write(line)
                    sampled_count += 1
                
                # Print progress every 100k lines
                if count % 100000 == 0:
                    print(f"Processed {count} lines... Saved {sampled_count}")

    print(f"Finished! Total: {count} | Sampled: {sampled_count}")

sample_dataset(input_file, output_file, sample_rate)

Processed 100000 lines... Saved 978
Processed 200000 lines... Saved 1943
Processed 300000 lines... Saved 2954
Processed 400000 lines... Saved 3942
Processed 500000 lines... Saved 5002
Processed 600000 lines... Saved 5978
Processed 700000 lines... Saved 6978
Processed 800000 lines... Saved 7968
Processed 900000 lines... Saved 8918
Processed 1000000 lines... Saved 9967
Processed 1100000 lines... Saved 10971
Processed 1200000 lines... Saved 11931
Processed 1300000 lines... Saved 12909
Processed 1400000 lines... Saved 13938
Processed 1500000 lines... Saved 14942
Processed 1600000 lines... Saved 15974
Processed 1700000 lines... Saved 16983
Processed 1800000 lines... Saved 17982
Processed 1900000 lines... Saved 18993
Processed 2000000 lines... Saved 20012
Processed 2100000 lines... Saved 21027
Processed 2200000 lines... Saved 21984
Processed 2300000 lines... Saved 22952
Processed 2400000 lines... Saved 23969
Processed 2500000 lines... Saved 24952
Processed 2600000 lines... Saved 25949
Proces

Read Meta

In [17]:
import gzip
import json
import pandas as pd
import os

def clean_and_filter_amazon_meta(input_path, output_csv, review_sample_path, chunk_size=10000):
    # 1. Load the sampled ASINs first to create a filter
    print(f"Loading ASINs from {review_sample_path} for filtering...")
    sample_df = pd.read_csv(
    review_sample_path, 
    header=None, 
    names=['user_id', 'parent_asin', 'rating', 'timestamp']
)
    valid_asins = set(sample_df['parent_asin'].unique())
    print(f"Filter loaded: {len(valid_asins)} unique products to keep.")

    # Ensure fresh start
    if os.path.exists(output_csv):
        os.remove(output_csv)

    records = []
    total_processed = 0
    total_kept = 0
    
    # 2. Process Metadata in chunks
    with gzip.open(input_path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                item = json.loads(line.strip())
                p_asin = item.get('parent_asin')

                # --- FILTERING STEP ---
                if p_asin not in valid_asins:
                    continue

                # --- CLEANING THE TEXT FIELDS ---
                for field in ['description', 'features', 'categories', 'title']:
                    val = item.get(field, "")
                    if isinstance(val, list):
                        text = " ".join([str(x) for x in val]).strip()
                        item[field] = text.replace('\n', ' ').replace('\r', '')
                    else:
                        item[field] = str(val).strip()

                # --- FLATTEN NESTED DETAILS ---
                details = item.pop('details', {})
                if isinstance(details, dict):
                    for k, v in details.items():
                        item[f"detail_{k}"] = str(v).strip()
                
                records.append(item)
                total_kept += 1
                
            except Exception:
                continue

            # --- CHUNK WRITING ---
            if len(records) >= chunk_size:
                write_chunk_to_csv(records, output_csv)
                records = [] 
                print(f"Read {i+1} lines... Kept {total_kept} matching products.")

        # Final flush
        if records:
            write_chunk_to_csv(records, output_csv)

    print(f"\nFinished! Cleaned and Filtered CSV saved as: {output_csv}")
    print(f"Total Kept: {total_kept}")

def write_chunk_to_csv(data_list, output_csv):
    df_chunk = pd.DataFrame(data_list)
    file_exists = os.path.isfile(output_csv)
    df_chunk.to_csv(output_csv, mode='a', index=False, header=not file_exists, encoding='utf-8')

# --- EXECUTION ---
REVIEW_SAMPLE = "samples/Electronics_1_percent.csv" # Your review sample
META_INPUT = "meta/meta_Electronics.jsonl.gz"
OUTPUT_FILE = "Electronics.csv"

clean_and_filter_amazon_meta(META_INPUT, OUTPUT_FILE, REVIEW_SAMPLE)

Loading ASINs from samples/Electronics_1_percent.csv for filtering...
Filter loaded: 75511 unique products to keep.
Read 116818 lines... Kept 10000 matching products.
Read 261348 lines... Kept 20000 matching products.
Read 424278 lines... Kept 30000 matching products.
Read 609341 lines... Kept 40000 matching products.
Read 812503 lines... Kept 50000 matching products.
Read 1029980 lines... Kept 60000 matching products.
Read 1252865 lines... Kept 70000 matching products.

Finished! Cleaned and Filtered CSV saved as: Electronics.csv
Total Kept: 75511


In [3]:
import pandas as pd
df = pd.read_csv("C:\\Users\Ronza\Dev\DP-MSD\\amazon\meta\meta_Health_and_Personal_Care.csv")

C:\Users\Ronza\AppData\Local\Temp\ipykernel_23580\584792190.py:2: DtypeWarning: Columns (16,17,18,20,26,31,34,37,42,51,52,53,64,65,66,68,70,73,76,78,79,80,83,86,88,89,90,91,92,94,95,97,98,99,100,101,102,106,107,109,112,113,114,117,118,119,120,121,123,125,126,127,128,129,130,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,175,176,177,178,179,180,181,182,183,184,186,187,188,189,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,220,222,223,224,225,226,227,228,230,231,232,233,237,238,239,240,241,242,243,245,246,247,248,249,250,251,252,253,254,255,256,258,261,262,263,264,265,266,267,269,270,271,272,273,274,275,276,277,280,281,283,284,285,286,287,289,290,291,292,293,296,297,298,299,300,301,303,305,311,312,313,314,315,316,318,319,320,321,322,323,325,326,328,329,331,332,333,335,336,337,338,339,341,342,343,344,346,3